# Customer Churn Analysis

**WHAT**: This notebook explores the IBM Telco Customer Churn dataset.
**WHY**: Before building machine learning models, we must understand our data, clean it, and discover patterns (EDA - Exploratory Data Analysis).
**HOW**: We will use `pandas` for data manipulation, and `matplotlib`/`seaborn` for visualization.
**INTERVIEW**: "In the first phase, I loaded the dataset, checked for missing values, corrected data types (like TotalCharges), and used visualizations to understand which features most strongly correlate with churn, such as contract type and tenure."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style
sns.set_theme(style="whitegrid")

# Ensure output directories exist
os.makedirs("../outputs/figures", exist_ok=True)

# Load data
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()

## 1. Data Understanding

In [ ]:
print(f"Dataset shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)

**Observation**: `TotalCharges` is an `object` (string) instead of a `float`. We need to fix this.

In [ ]:
# Check for missing values
print(df.isnull().sum())

## 2. Data Cleaning

In [ ]:
# Convert TotalCharges to numeric, coercing errors to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many NaNs were created
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Since it's only 11 rows out of 7043, we can safely drop them
df.dropna(subset=['TotalCharges'], inplace=True)

# Drop customerID as it has no predictive power
df.drop('customerID', axis=1, inplace=True)

# Convert Target 'Churn' to binary (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Cleaned dataset shape: {df.shape}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
def save_and_show(fig, filename):
    fig.savefig(f"../outputs/figures/{filename}", bbox_inches="tight")
    plt.show()

### 3.1 Churn Distribution

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.xticks([0, 1], ['No Churn (0)', 'Churn (1)'])
save_and_show(plt.gcf(), 'churn_distribution.png')

**Insight**: The dataset is imbalanced. There are more non-churners than churners.

### 3.2 Churn by Contract Type

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Churn by Contract Type')
save_and_show(plt.gcf(), 'churn_by_contract.png')

**Insight**: Customers on a 'Month-to-month' contract have a significantly higher churn rate compared to those on 1-year or 2-year contracts.

### 3.3 Churn by Tenure (Months)

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True)
plt.title('Churn by Tenure')
save_and_show(plt.gcf(), 'churn_by_tenure.png')

**Insight**: Newer customers (low tenure) are much more likely to churn.

### 3.4 Churn by Monthly Charges

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True)
plt.title('Churn by Monthly Charges')
save_and_show(plt.gcf(), 'churn_by_monthly_charges.png')

**Insight**: Higher monthly charges generally correlate with higher churn.

### 3.5 Numerical Correlation

In [ ]:
plt.figure(figsize=(8, 6))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
save_and_show(plt.gcf(), 'correlation_matrix.png')

**Conclusion**: We have identified key factors for churn: Month-to-month contracts, low tenure, and high monthly charges. The data is clean and ready for feature engineering.